# Checkpoint 2 Notebook: Research Questions and Initial Methods

This notebook extends checkpoint 1 with additional exploratory analysis, question formation, and first-pass modeling experiments.

In [ ]:
# Reproducibility Header (Run this first)
# Run order note: use "Run All" from top to ensure reproducible outputs.

RANDOM_SEED = 42

import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

DATA_DIR = "data/csv_data"
YEARS = list(range(2000, 2027))
CSV_PATHS = [os.path.join(DATA_DIR, f"atp_{year}.csv") for year in YEARS]

print(f"Random seed: {RANDOM_SEED}")
print(f"Number of input files configured: {len(CSV_PATHS)}")
print("Run order: Run All from top")

## 1) Project Scope Recap

- **Domain**: ATP men's professional tennis match outcomes.
- **Target goal**: build interpretable and predictive analyses around match winners and contextual factors.
- **Data source**: yearly ATP CSV files (`data/csv_data/atp_YYYY.csv`).

> **Decision Log (Scope):** We keep the same domain and dataset family from checkpoint 1 to ensure continuity and reduce integration risk; this checkpoint focuses on stronger RQ design and methodological grounding rather than changing data provenance.

In [ ]:
# Load and combine yearly ATP match files
frames = []
for path in CSV_PATHS:
    if os.path.exists(path):
        tmp = pd.read_csv(path)
        tmp["source_file"] = os.path.basename(path)
        frames.append(tmp)

if not frames:
    raise FileNotFoundError("No ATP CSV files were found under data/csv_data/.")

df = pd.concat(frames, ignore_index=True)
print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")
df.head(3)

## 2) Additional EDA for RQ Formation

> **Decision Log (EDA design):** We selected lightweight, high-signal EDA slices (missingness, rank distributions, and surface-level win context) because they directly inform feasible questions and candidate feature sets without overfitting to one tournament or year.

In [ ]:
# Quick structural EDA
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
missing_pct.head(15)

In [ ]:
# Rank-focused EDA
rank_cols = [c for c in ["winner_rank", "loser_rank", "winner_rank_points", "loser_rank_points"] if c in df.columns]
if rank_cols:
    display(df[rank_cols].describe().T)
else:
    print("Rank columns not found in this dataset version.")

In [ ]:
# Surface distribution for context
if "surface" in df.columns:
    plt.figure(figsize=(8, 4))
    order = df["surface"].value_counts().index
    sns.countplot(data=df, x="surface", order=order)
    plt.title("Match count by surface")
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()
else:
    print("Column 'surface' not found.")

## 3) Research Questions

1. **RQ1 (Predictive):** How well can pre-match structured attributes predict whether a higher-ranked player wins?
2. **RQ2 (Contextual):** How does predictive signal differ across court surfaces (Hard, Clay, Grass, Carpet)?
3. **RQ3 (Interpretability):** Which pre-match features (rank gap, points gap, surface, round) contribute most to model predictions?

> **Decision Log (RQ definition):** We constrained RQs to questions answerable from pre-match fields so our modeling setup avoids post-match leakage and remains methodologically valid.

## 4) Motivation & Feasibility

- **Motivation:** Accurate and interpretable pre-match outcome modeling is useful for understanding competitive structure in ATP events.
- **Feasibility:** The dataset is longitudinal, structured, and large enough for train/test evaluation plus subgroup analysis by surface.
- **Risk controls:** We explicitly avoid leakage fields (e.g., winner/loser-specific post-match stats) when creating features.

> **Decision Log (Feasibility):** We chose binary framing (higher-ranked wins vs. upset) as a tractable first target that supports both baseline and ensemble methods with straightforward evaluation metrics.

## 5) Methodological Plan

Planned workflow:
1. Build a binary label `higher_rank_wins` from winner/loser ranks.
2. Engineer pre-match features (rank and points gaps, categorical context like surface/round).
3. Train a regularized Logistic Regression baseline and a Random Forest nonlinear comparator.
4. Evaluate with Accuracy and ROC-AUC; inspect feature importance/proxy importance.
5. Repeat evaluation stratified by surface for RQ2.

> **Decision Log (Methods):** Logistic Regression was selected as a transparent baseline; Random Forest was selected to capture nonlinear interactions with minimal feature scaling burden.

In [ ]:
# Build target and features for initial runs
working = df.copy()
required = ["winner_rank", "loser_rank"]
for col in required:
    if col not in working.columns:
        raise KeyError(f"Required column missing: {col}")

working = working.dropna(subset=required).copy()
working["higher_rank_wins"] = (working["winner_rank"] < working["loser_rank"]).astype(int)
working["rank_gap"] = working["loser_rank"] - working["winner_rank"]

if "winner_rank_points" in working.columns and "loser_rank_points" in working.columns:
    working["points_gap"] = working["winner_rank_points"] - working["loser_rank_points"]
else:
    working["points_gap"] = np.nan

candidate_features = ["rank_gap", "points_gap", "surface", "round", "best_of"]
feature_cols = [c for c in candidate_features if c in working.columns]

X = working[feature_cols]
y = working["higher_rank_wins"]

num_features = [c for c in feature_cols if pd.api.types.is_numeric_dtype(X[c])]
cat_features = [c for c in feature_cols if c not in num_features]

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median"))
        ]), num_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_features),
    ],
    remainder="drop"
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)

print("Training rows:", len(X_train), "| Test rows:", len(X_test))
print("Features used:", feature_cols)

## 6) Initial Method Runs

> **Decision Log (Initial hyperparameters):**
> - Logistic Regression: `C=1.0`, `max_iter=1000`, `class_weight='balanced'` for a stable, regularized baseline under possible class imbalance.
> - Random Forest: `n_estimators=300`, `max_depth=12`, `min_samples_leaf=5`, `class_weight='balanced_subsample'` to reduce overfitting while preserving nonlinear flexibility.

In [ ]:
# Model 1: Logistic Regression
log_reg = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(C=1.0, max_iter=1000, class_weight="balanced", random_state=RANDOM_SEED))
])

log_reg.fit(X_train, y_train)
log_pred = log_reg.predict(X_test)
log_prob = log_reg.predict_proba(X_test)[:, 1]

print("Logistic Regression")
print("  Accuracy:", round(accuracy_score(y_test, log_pred), 4))
print("  ROC-AUC :", round(roc_auc_score(y_test, log_prob), 4))

In [ ]:
# Model 2: Random Forest
rf = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_depth=12,
        min_samples_leaf=5,
        class_weight="balanced_subsample",
        random_state=RANDOM_SEED,
        n_jobs=-1
    ))
])

rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:, 1]

print("Random Forest")
print("  Accuracy:", round(accuracy_score(y_test, rf_pred), 4))
print("  ROC-AUC :", round(roc_auc_score(y_test, rf_prob), 4))

In [ ]:
# Surface-level slice for RQ2 (illustrative)
if "surface" in X_test.columns:
    results = []
    eval_df = X_test.copy()
    eval_df["y_true"] = y_test.values
    eval_df["log_prob"] = log_prob
    eval_df["rf_prob"] = rf_prob

    for surface_name, g in eval_df.groupby("surface"):
        if g["y_true"].nunique() < 2:
            continue
        log_auc = roc_auc_score(g["y_true"], g["log_prob"])
        rf_auc = roc_auc_score(g["y_true"], g["rf_prob"])
        results.append((surface_name, len(g), log_auc, rf_auc))

    surface_perf = pd.DataFrame(results, columns=["surface", "n", "log_auc", "rf_auc"]).sort_values("n", ascending=False)
    display(surface_perf)
else:
    print("Surface not available for stratified evaluation.")

## 7) RQ-to-Method Mapping Table

| Research Question | Data slice | Primary method | Output artifact |
|---|---|---|---|
| RQ1: Predict higher-ranked win | Full dataset split | Logistic Regression + Random Forest | Accuracy/ROC-AUC comparison |
| RQ2: Surface differences | Test data grouped by `surface` | Stratified AUC analysis | Surface performance table |
| RQ3: Feature contribution | Trained models | Coefficients / feature importances | Ranked feature report |

> **Decision Log (Mapping):** We assigned at least one concrete, measurable artifact per RQ to keep evaluation objective and checkpoint deliverables auditable.

## 8) Collaboration Declaration

- Team members collaborated on scope alignment, feature brainstorming, and method selection.
- Notebook assembly and baseline implementation were completed jointly, with shared review before submission.
- All contributors reviewed the final RQs and method mapping for consistency.

> **Decision Log (Collaboration):** We used role separation (EDA lead, modeling lead, reviewer) to reduce duplication and improve quality control under checkpoint time constraints.